#  Storm Prediction — Pipeline Front

Ce notebook lance le pipeline complet de `storm_prediction` :
1. **Vérification des dépendances**
2. **Configuration** (chemins, paramètres)
3. **Lancement** du batch ou d'un orage unique
4. **Résultats** : stats + lien vers `index.html`

## 1. Dépendances

In [47]:
import subprocess, sys

required = ["pandas", "numpy"]
for pkg in required:
    try:
        __import__(pkg)
        
        print(f"{pkg}")
    except ImportError:
        print(f"Installation de {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
        print(f"{pkg} installé")

pandas
numpy


## 2. Configuration

Modifie les variables ci-dessous selon tes besoins.

In [48]:
from pathlib import Path

# ─── Chemins ───────────────────────────────────────────────────────────────
SCRIPT_DIR  = Path("src/test_direction")
SCRIPT_PATH = SCRIPT_DIR / "storm_direction_analysis.py"
CSV_PATH    = Path("data/raw/data_with_storm_id.csv")
OUTPUT_DIR  = SCRIPT_DIR / "output"

# ─── Paramètres d'exécution ────────────────────────────────────────────────
STORM_ID          = None    # None = batch complet | ex: "Ajaccio_0068" = un seul orage
MIN_DURATION      = 1       # Durée minimale d'un orage en minutes
LIMIT_PER_AIRPORT = 50      # 50 orages par ville → estimations plus représentatives

# ─── Vérifications ─────────────────────────────────────────────────────────
print(f"Script   : {SCRIPT_PATH.resolve()}")
print(f"Script   : {' trouvé' if SCRIPT_PATH.exists() else 'INTROUVABLE'}")

csv_resolved = CSV_PATH.resolve()
print(f"CSV      : {csv_resolved}")
print(f"CSV      : {'trouvé' if csv_resolved.exists() else 'INTROUVABLE'}")

print(f"Output   : {OUTPUT_DIR.resolve()}")
mode = ("Orage unique → " + STORM_ID) if STORM_ID else f"Batch ({LIMIT_PER_AIRPORT} orages/ville)"
print(f"Mode     : {mode}")


Script   : /home/alvi/Documents/Github/databattle2026-meteorage/storm_prediction/src/test_direction/storm_direction_analysis.py
Script   :  trouvé
CSV      : /home/alvi/Documents/Github/databattle2026-meteorage/storm_prediction/data/raw/data_with_storm_id.csv
CSV      : trouvé
Output   : /home/alvi/Documents/Github/databattle2026-meteorage/storm_prediction/src/test_direction/output
Mode     : Batch (50 orages/ville)


## 3. Lancement du pipeline

In [49]:
import subprocess, sys, time

cmd = [
    sys.executable,
    str(SCRIPT_PATH.resolve()),
    "--csv",    str(CSV_PATH.resolve()),
    "--output", str(OUTPUT_DIR.resolve()),
    "--min_duration", str(MIN_DURATION),
]

if STORM_ID:
    cmd += ["--storm_id", STORM_ID]
elif LIMIT_PER_AIRPORT:
    cmd += ["--limit_per_airport", str(LIMIT_PER_AIRPORT)]

print(" Commande :", " ".join(cmd))
print("-" * 60)

t0 = time.time()
result = subprocess.run(
    cmd,
    cwd=str(SCRIPT_DIR.resolve()),
    capture_output=False,
    text=True
)

elapsed = time.time() - t0
print("-" * 60)
if result.returncode == 0:
    print(f"Pipeline terminé en {elapsed:.1f}s")
else:
    print(f"Erreur (code {result.returncode}) en {elapsed:.1f}s")


 Commande : /home/alvi/Documents/deep learning/segmentation_env/bin/python /home/alvi/Documents/Github/databattle2026-meteorage/storm_prediction/src/test_direction/storm_direction_analysis.py --csv /home/alvi/Documents/Github/databattle2026-meteorage/storm_prediction/data/raw/data_with_storm_id.csv --output /home/alvi/Documents/Github/databattle2026-meteorage/storm_prediction/src/test_direction/output --min_duration 1 --limit_per_airport 50
------------------------------------------------------------
Chargement de /home/alvi/Documents/Github/databattle2026-meteorage/storm_prediction/data/raw/data_with_storm_id.csv...
Limite par aeroport (50) : 250 orages au total
   Ajaccio: 50
   Bastia: 50
   Biarritz: 50
   Nantes: 50
   Pise: 50
250 orages a analyser (duree >= 1.0 min)

  → /home/alvi/Documents/Github/databattle2026-meteorage/storm_prediction/src/test_direction/output/Ajaccio_0002.html
  Ajaccio_0002: 42.4 km/h -> E (25.35 km)
  [skip] Ajaccio_0006: pas assez d'eclairs (2)
  → /hom

## 4. Résultats

In [50]:
import json
import pandas as pd
from IPython.display import display, HTML

# ─── Stats fichiers générés ────────────────────────────────────────────────
html_files = list(OUTPUT_DIR.glob("*.html"))
html_storms = [f for f in html_files if f.name != "index.html"]

print(f"Fichiers générés dans {OUTPUT_DIR.resolve()}")
print(f"   {len(html_storms)} orages HTML + {'index.html ' if (OUTPUT_DIR / 'index.html').exists() else 'index.html '}")

# ─── Lecture du JSON de résultats ─────────────────────────────────────────
json_path = OUTPUT_DIR / "storm_directions.json"
if json_path.exists():
    with open(json_path) as f:
        raw = json.load(f)

    if isinstance(raw, list):
        entries = [(item.get("storm_id", f"#{i}"), item) for i, item in enumerate(raw)]
    else:
        entries = list(raw.items())

    rows = []
    for storm_id, info in entries:
        r = info.get("ransac") or {}
        conf = r.get("confidence")
        inlier = r.get("inlier_ratio")
        az = r.get("r2_azimut")
        rows.append({
            "ID":           storm_id,
            "Aéroport":     info.get("airport", ""),
            "Direction":    r.get("direction", ""),
            "Vitesse km/h": round(r.get("speed_kmh", 0), 1) if r.get("speed_kmh") else None,
            "Conf %":       round(conf * 100) if conf is not None else None,
            "Inliers %":    round(inlier * 100) if inlier is not None else None,
            "az R²%":       round(az * 100) if az is not None else None,
            "Prédiction":   "" if info.get("predicted_exit") else "—",
        })

    df = pd.DataFrame(rows)
    for col in ["Conf %", "Inliers %", "az R²%", "Vitesse km/h"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # ─── Stats par ville ───────────────────────────────────────────────────
    print(f"\nRésultats RANSAC par ville ({LIMIT_PER_AIRPORT} orages analysés/ville) :\n")
    df_valid = df[df["Conf %"].notna() & df["Inliers %"].notna()]

    stats_rows = []
    for airport, grp in df_valid.groupby("Aéroport"):
        n_total = len(grp)
        n_pred = (grp["Prédiction"] == "").sum()
        stats_rows.append({
            "Aéroport":        airport,
            "Orages analysés": n_total,
            "Avec prédiction": f"{n_pred} ({round(n_pred/n_total*100)}%)",
            "Inliers moy.":    f"{grp['Inliers %'].mean():.0f}%",
            "Confiance moy.":  f"{grp['Conf %'].mean():.0f}%",
            "az R² moy.":      f"{grp['az R²%'].mean():.0f}%",
            "Vitesse moy.":    f"{grp['Vitesse km/h'].mean():.0f} km/h",
        })

    df_stats = pd.DataFrame(stats_rows).set_index("Aéroport")
    display(df_stats)

    # ─── Top 10 ────────────────────────────────────────────────────────────
    print(f"\nTop 10 orages (meilleure confiance RANSAC) :")
    top10 = df_valid.sort_values("Conf %", ascending=False).head(10).reset_index(drop=True)
    display(top10[["ID", "Aéroport", "Direction", "Vitesse km/h", "Inliers %", "Conf %", "az R²%", "Prédiction"]])

else:
    print("storm_directions.json introuvable — lancez d'abord le pipeline (cellule 3)")


Fichiers générés dans /home/alvi/Documents/Github/databattle2026-meteorage/storm_prediction/src/test_direction/output
   2246 orages HTML + index.html 

Résultats RANSAC par ville (50 orages analysés/ville) :



,Orages analysés,Avec prédiction,Inliers moy.,Confiance moy.,az R² moy.,Vitesse moy.
Aéroport,,,,,,
Ajaccio,39,38 (97%),51%,31%,63%,38 km/h
Bastia,41,41 (100%),44%,25%,59%,23 km/h
Biarritz,28,28 (100%),51%,39%,74%,68 km/h
Nantes,35,35 (100%),51%,32%,66%,50 km/h
Pise,23,21 (91%),43%,20%,35%,50 km/h



Top 10 orages (meilleure confiance RANSAC) :


,ID,Aéroport,Direction,Vitesse km/h,Inliers %,Conf %,az R²%,Prédiction
0,Biarritz_0042,Biarritz,S,37.6,100.0,93.0,97.0,
1,Biarritz_0058,Biarritz,E,43.7,100.0,92.0,95.0,
2,Nantes_0044,Nantes,NE,89.1,100.0,90.0,90.0,
3,Nantes_0040,Nantes,N,73.3,100.0,88.0,82.0,
4,Ajaccio_0012,Ajaccio,NE,84.6,75.0,75.0,100.0,—
5,Biarritz_0025,Biarritz,E,248.3,75.0,72.0,93.0,
6,Nantes_0005,Nantes,S,147.5,80.0,72.0,86.0,
7,Ajaccio_0041,Ajaccio,SO,9.4,100.0,68.0,78.0,
8,Ajaccio_0045,Ajaccio,SE,25.9,67.0,62.0,88.0,
9,Nantes_0035,Nantes,NE,50.9,75.0,61.0,83.0,


In [52]:
import sys, subprocess
from IPython.display import display, HTML

index_path = OUTPUT_DIR / "index.html"
if index_path.exists():
    abs_path = index_path.resolve()

    # Commande d'ouverture selon l'OS
    if sys.platform == "darwin":
        open_cmd = f"open {abs_path}"
    elif sys.platform == "win32":
        open_cmd = f"start {abs_path}"
    else:
        open_cmd = f"xdg-open {abs_path}"

    display(HTML(f"""
    <div style="padding:12px; background:#1e1e2e; borderm-radius:8px; border:1px solid #313244;">
        <h3 style="color:#cdd6f4; margin:0 0 8px 0;">🗺️ Visualisation interactive</h3>
        <p style="color:#a6adc8; margin:0 0 12px 0;">Ouvre le fichier suivant dans ton navigateur :</p>
        <code style="background:#313244; color:#89b4fa; padding:8px 14px; border-radius:6px; display:block; font-size:13px;">
            {abs_path}
        </code>
        <p style="color:#a6adc8; margin:12px 0 0 0; font-size:12px;">
            💡 Ou depuis un terminal : <code style="color:#a6e3a1;">{open_cmd}</code>
        </p>
    </div>
    """))

    # Ouvrir automatiquement dans le navigateur
    if sys.platform == "darwin":
        subprocess.Popen(["open", str(abs_path)])
    elif sys.platform == "win32":
        subprocess.Popen(["start", str(abs_path)], shell=True)
    else:
        subprocess.Popen(["xdg-open", str(abs_path)])
else:
    print("index.html non trouvé — lancez d'abord le pipeline (cellule 3)")


Gtk-Message: 15:00:26.559: Not loading module "atk-bridge": The functionality is provided by GTK natively. Please try to not load it.
